# 3D DC Inversion — SimPEG vanilla vs PyTorch (cuDSS): head-to-head

Reproduction of the SimPEG `inv_dcr_3d` tutorial — **same real topography, same survey geometry,
same two-sphere model** (conductor + resistor) — all in one notebook, comparing:

**Vanilla** — SimPEG **Gauss-Newton + Pardiso MKL** (matrix-free, see §4).
**PyTorch** — **L-BFGS + cuDSS GPU** (differentiable forward, exact autograd gradient), same L2 prior.

The tutorial's topography (`topo_xyz.txt`) and survey (`dc_data.xyz`) are downloaded from the SimPEG
user-tutorials assets. Data are re-synthesized by forward-modelling the two spheres on our mesh (clean
+ noise), so both inversions invert the exact same `dobs`.

Fair CPU baseline = Pardiso MKL (parallel), NOT SimPEG's default scipy SuperLU.

In [ ]:
import os, time, tarfile, urllib.request
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import torch

from pydiso.mkl_solver import MKLPardisoSolver as _MKL
from SimPEG.config import SimpegConfig
from SimPEG import (maps, data, data_misfit, regularization, optimization,
                    inverse_problem, inversion, directives)
from SimPEG.electromagnetics.static import resistivity as dc
from SimPEG.utils.io_utils.io_utils_electromagnetics import read_dcip_xyz
from discretize import TreeMesh
from discretize.utils import active_from_xyz

cfg = SimpegConfig(); cfg.dtype = torch.float64
HAS_CUDA = torch.cuda.is_available()
GPU_SOLVER = "cudss" if HAS_CUDA else "pcg_gpu"
print("CUDA:", HAS_CUDA, "| GPU solver:", GPU_SOLVER)

class PardisoSolver:                       # fair parallel CPU baseline
    def __init__(s, A, **k): s._s = _MKL(sp.csr_matrix(A))
    def __mul__(s, b): return s._s.solve(np.asarray(b))
    def solve(s, b, **k): return s._s.solve(np.asarray(b))
    def clean(s): pass

## 0. ipywidgets check (run this FIRST)

Verifies the interactive viewer (section 9) works before spending time on the inversions.

In [ ]:
HAS_WIDGETS = False
try:
    import ipywidgets as widgets
    from IPython.display import display
    _log = []
    _s = widgets.IntSlider(min=0, max=5, value=1)
    _s.observe(lambda ch: _log.append(ch["new"]), names="value")
    _s.value = 4
    assert _log == [4], f"widget callback did not fire: {_log}"
    HAS_WIDGETS = True
    print("ipywidgets", widgets.__version__, "- callback OK", _log)
    _out = widgets.Output()
    def _probe(k=2):
        with _out: _out.clear_output(); print(f"slider works -> k={k}, k^2={k**2}")
    display(widgets.interactive(_probe, k=widgets.IntSlider(min=0, max=5, value=2, description="test")), _out)
    print("OK: move the slider above; if the text changes, section 9 will work.")
except Exception as e:
    print("ipywidgets NOT usable:", repr(e))
    print("Fix: pip install ipywidgets  then restart the kernel. Viewer falls back to a static plot.")

## 1. Parameters

In [ ]:
DH = 35.0            # base cell (m). 25 == exact tutorial (~71k active, heavier VRAM);
                     # 35 -> ~46k (moderate, default); 50 -> ~28k (fast). Spheres r=165 stay well resolved.
# --- vanilla side = exact tutorial recipe (so the comparison is apples-to-apples) ---
NOISE = 0.10                            # 10% Gaussian noise == tutorial's 10% uncertainty -> chifact=1 is meaningful
GN_MAXITER, GN_MAXITERCG, GN_TOLCG = 40, 30, 1e-3   # tutorial: maxIter=40, cg_maxiter=30, cg_rtol=1e-3
LENGTH_SCALE = 100.0                    # tutorial: length_scale_x/y/z = 100 m
COOL = 2.0                              # BetaSchedule coolingFactor (shared by both sides)
COOLING_RATE = 2                        # cool beta every COOLING_RATE iters (tutorial coolingRate=2) -> same on both sides
# --- pytorch side ---
LBFGS_OUTER, LBFGS_MAXITER = 16, 15
FORCE_ITERS = True   # force exactly LBFGS_OUTER iterations in the pytorch inversion (no early stop)

## 2. Real topography + survey, tutorial mesh, two-sphere model (+ plotting helpers)

Downloads the tutorial's `topo_xyz.txt` (hilly topography, z 311-409 m) and `dc_data.xyz`
(6 dipole-dipole lines, 696 data) once into `inv_dcr_3d_files/`. The mesh is built with the
tutorial's recipe (origin shifted to topo max, `refine_surface` on topography + `refine_points`
on electrodes). True model: background 100, conductor 10 at (-300,0,100), resistor 1000 at
(+300,0,100) ohm-m, spheres r=165 m — identical to the tutorial.

In [ ]:
DATA_URL = "https://github.com/simpeg/user-tutorials/raw/main/assets/05-dcr/inv_dcr_3d_files.tar.gz"
DATA_DIR = "inv_dcr_3d_files"
topo_file = os.path.join(DATA_DIR, "topo_xyz.txt"); data_file = os.path.join(DATA_DIR, "dc_data.xyz")
if not (os.path.isfile(topo_file) and os.path.isfile(data_file)):
    tgz = "inv_dcr_3d_files.tar.gz"; print("downloading tutorial topography + survey ...")
    urllib.request.urlretrieve(DATA_URL, tgz)
    with tarfile.open(tgz, "r:gz") as t: t.extractall(DATA_DIR)
topo_xyz = np.loadtxt(topo_file)
TOPO_MAX = topo_xyz[:, -1].max()

# survey geometry from the real file (we ignore its V/A values and re-synthesize dobs in section 3)
_dc_in, _ = read_dcip_xyz(data_file, "volt", data_header="V/A", uncertainties_header="UNCERT",
                          is_surface_data=False, dict_headers=["LINEID"])
survey = _dc_in.survey

# tutorial mesh recipe: 8000 x 8000 x 4000 domain, origin to topo max, refine surface + electrodes
DWX = DWY = 8000.0; DWZ = 4000.0
nb_ = lambda w: 2**int(np.round(np.log(w/DH)/np.log(2.0)))
mesh = TreeMesh([[(DH,nb_(DWX))],[(DH,nb_(DWY))],[(DH,nb_(DWZ))]], x0="CCN", diagonal_balance=True)
mesh.origin = mesh.origin + np.r_[0.0, 0.0, TOPO_MAX]
k = np.sqrt(np.sum(topo_xyz[:,0:2]**2, axis=1)) < 1200
mesh.refine_surface(topo_xyz[k,:], padding_cells_by_level=[0,4,4], finalize=False)
mesh.refine_points(survey.unique_electrode_locations, padding_cells_by_level=[6,6,4], finalize=False)
mesh.finalize()
active = active_from_xyz(mesh, topo_xyz); nC = int(active.sum())
cc = mesh.cell_centers[active,:]
sig = 1e-2*np.ones(nC)                                                  # background  100 ohm-m
sig[((cc[:,0]+300)**2+cc[:,1]**2+(cc[:,2]-100)**2) < 165**2] = 1e-1     # conductor   10 ohm-m  (-300,0,100)
sig[((cc[:,0]-300)**2+cc[:,1]**2+(cc[:,2]-100)**2) < 165**2] = 1e-3     # resistor    1000 ohm-m (+300,0,100)
m_true = np.log(1/sig)                                                  # model = log-resistivity
rmap = maps.InjectActiveCells(mesh, active, 1/1e-8)*maps.ExpMap(nP=nC)
survey.drape_electrodes_on_topography(mesh, active, option="top")
VMIN, VMAX = 8.0, 1300.0
XLIM = [-1200, 1200]; ZLIM = [TOPO_MAX-1200, TOPO_MAX]
print(f"mesh {mesh.n_cells} | active {nC} | nD {survey.nD} | topo max {TOPO_MAX:.1f} m")

# shared plotting / metric helpers (used by sections 7, 8, 9)
plotting_map = maps.InjectActiveCells(mesh, active, np.nan)
def errf(mlog): return float(np.linalg.norm(mlog-m_true)/np.linalg.norm(m_true))   # rel error to true
def pslice(ax, mlog, title):
    mesh.plot_slice(plotting_map*np.exp(mlog), ax=ax, normal="Y", ind=int(len(mesh.h[1])/2),
        grid=False, pcolor_opts={"cmap":"RdYlBu_r","norm":LogNorm(vmin=VMIN,vmax=VMAX)})
    ax.set_title(title); ax.set_xlim(XLIM); ax.set_ylim(ZLIM); ax.set_xlabel("x (m)"); ax.set_ylabel("z (m)")

## 3. Synthetic data + regularization

In [ ]:
cfg.torch_is_active = False
simd = dc.simulation.Simulation3DNodal(mesh, survey=survey, rhoMap=rmap)
dclean = simd.dpred(m_true)
np.random.seed(1)
dobs = dclean + NOISE*np.abs(dclean)*np.random.randn(len(dclean))
std = 1e-7 + 0.1*np.abs(dobs)              # tutorial uncertainty: 1e-7 V/A + 10%
dcd = data.Data(survey, dobs=dobs); dcd.standard_deviation = std
mref = np.log(100.0)*np.ones(nC); nD = survey.nD; target = float(nD)   # TargetMisfit chifact=1

reg = regularization.WeightedLeastSquares(mesh, active_cells=active,
        length_scale_x=LENGTH_SCALE, length_scale_y=LENGTH_SCALE, length_scale_z=LENGTH_SCALE,
        reference_model=mref)
R = sp.csr_matrix(reg.deriv2(mref))                 # L2 Hessian (constant) for the pytorch loss
print(f"nD {nD} | target phi_d {target:.0f}")

## 4. Vanilla — Gauss-Newton + Pardiso (`storeJ=True`, fast path)

Uses SimPEG's **explicit-Jacobian fast path** (`storeJ=True`): J is formed & factored-reuse once per
GN iteration, so the inner CG steps are cheap dense mat-vecs (no per-CG forward/adjoint solves). We add
the **Jacobi preconditioner** (`UpdatePreconditioner`) — it speeds up CG **without changing the
objective**. We deliberately **omit sensitivity weights** so both sides minimize the *exact same* L2
objective (`f = φ_d + β·φ_m`, same R) — true apples-to-apples isolating optimizer + hardware.

(The fork's `getJ` Robin-BC term used to mis-broadcast on the multi-RHS full-J path; fixed 2026-06 by
routing it through the shared `_inner_mat_mul_op` helper, so `storeJ=True` now works and matches the
matrix-free Jacobian to 1e-15.)

In [ ]:
class SaveIters(directives.InversionDirective):
    """Capture model + predicted data each iteration (for the viewer / error curve)."""
    def _sim(self):
        dm = self.invProb.dmisfit
        return getattr(dm, "simulation", None) or dm.objfcts[0].simulation
    def initialize(self):
        import time as _t; self.t0 = _t.time(); self.times = [0.0]
        m0 = np.array(self.invProb.model).copy()
        self.models = [m0]; self.dpreds = [np.array(self._sim().dpred(m0)).copy()]
    def endIter(self):
        import time as _t
        m = np.array(self.opt.xc).copy()
        self.models.append(m); self.dpreds.append(np.array(self._sim().dpred(m)).copy())
        self.times.append(_t.time()-self.t0)

def run_gn():
    cfg.torch_is_active = False
    sim = dc.simulation.Simulation3DNodal(mesh, survey=survey, rhoMap=rmap, storeJ=True)  # fast path
    sim.solver = PardisoSolver
    dmis = data_misfit.L2DataMisfit(data=dcd, simulation=sim)
    opt = optimization.InexactGaussNewton(maxIter=GN_MAXITER, maxIterCG=GN_MAXITERCG, tolCG=GN_TOLCG, maxIterLS=20)
    invProb = inverse_problem.BaseInvProblem(dmis, reg, opt)
    save = directives.SaveOutputEveryIteration(save_txt=False); saveit = SaveIters()
    dlist = [directives.UpdatePreconditioner(update_every_iteration=True),   # speeds CG, same objective (no SW)
             directives.BetaEstimate_ByEig(beta0_ratio=100.0), save, saveit,
             directives.BetaSchedule(coolingFactor=COOL, coolingRate=COOLING_RATE),
             directives.TargetMisfit(chifact=1.0)]
    inv = inversion.BaseInversion(invProb, directiveList=dlist)
    t0 = time.time(); mrec = inv.run(mref); dt = time.time()-t0
    return dict(model=mrec, time=dt, phi_d=np.array(save.phi_d), times=np.array(saveit.times),
                models=saveit.models, dpreds=saveit.dpreds)

res_gn = run_gn()
print(f"GN-Pardiso: {res_gn['time']:.1f}s | {len(res_gn['phi_d'])} iters | phi_d {res_gn['phi_d'][-1]:.1f} | err {errf(res_gn['model']):.3f}")

## 5. PyTorch — L-BFGS + cuDSS (same L2 prior)

In [ ]:
def s2t(M):
    M = M.tocoo()
    return torch.sparse_coo_tensor(torch.tensor(np.vstack((M.row,M.col))),
                                   torch.tensor(M.data, dtype=torch.float64), M.shape).coalesce()
R_t = s2t(R)
mref_t = torch.tensor(mref); W_t = torch.tensor(1.0/std); dobs_t = torch.tensor(dobs)

def phi_m(m):                                  # weighted L2 prior (mesh-aware Hessian R)
    d2 = (m - mref_t).unsqueeze(1)
    return 0.5*torch.sum(d2*torch.sparse.mm(R_t, d2))

def run_lbfgs():
    cfg.torch_is_active = True; cfg.device = ("cuda" if HAS_CUDA else "cpu")
    cfg.dtype = torch.float64; cfg.solver = GPU_SOLVER
    sim = dc.simulation.Simulation3DNodal(mesh, survey=survey, rhoMap=rmap)
    m = torch.tensor(mref.copy(), requires_grad=True)             # CPU model; solve on GPU
    with torch.no_grad():
        d0 = sim.dpred(mref_t, f=sim.fields(mref_t)); d0 = d0.cpu() if d0.is_cuda else d0
        phid0 = 0.5*torch.sum((W_t*(d0-dobs_t))**2)
        pm = phi_m(mref_t + 0.1*torch.randn(nC, dtype=torch.float64))
    beta = float(phid0/(pm+1e-30))
    opt = torch.optim.LBFGS([m], lr=1.0, max_iter=LBFGS_MAXITER, history_size=10, line_search_fn="strong_wolfe")
    hist = []; times = []; models = [mref.copy()]; dpreds = [(d0.cpu() if d0.is_cuda else d0).detach().numpy()]
    if HAS_CUDA: torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for outer in range(LBFGS_OUTER):
        pd = [None]
        def closure():
            opt.zero_grad()
            m_in = m*1.0
            d = sim.dpred(m_in, f=sim.fields(m_in)); d = d.cpu() if d.is_cuda else d
            loss = 0.5*torch.sum((W_t*(d-dobs_t))**2) + beta*phi_m(m)
            loss.backward(); pd[0] = float(0.5*torch.sum((W_t*(d-dobs_t))**2)); return loss
        opt.step(closure); hist.append(pd[0]); times.append(time.time()-t0)   # cumulative wall time / outer
        with torch.no_grad():
            da = sim.dpred(m*1.0, f=sim.fields(m*1.0)); da = da.cpu() if da.is_cuda else da
        models.append(m.detach().cpu().numpy().copy()); dpreds.append(da.numpy())
        if (not FORCE_ITERS) and pd[0] <= target: break
        if (outer+1) % COOLING_RATE == 0: beta /= COOL    # same cooling cadence as vanilla BetaSchedule
    dt = time.time()-t0
    vram = torch.cuda.max_memory_allocated()/1e9 if HAS_CUDA else 0.0
    cfg.torch_is_active = False
    return dict(model=m.detach().cpu().numpy(), time=dt, phi_d=np.array(hist), times=np.array(times),
                vram=vram, models=models, dpreds=dpreds)

print("L-BFGS + cuDSS ..."); res_pt = run_lbfgs()
print(f"  {res_pt['time']:.1f}s | outer {len(res_pt['phi_d'])} | phi_d {res_pt['phi_d'][-1]:.1f} | VRAM {res_pt['vram']:.2f} GB | err {errf(res_pt['model']):.3f}")

## 6. Speed summary

**Raw** = total wall time each scheme ran. This is *unfair* to the GPU because `FORCE_ITERS=True`
makes pytorch run all `LBFGS_OUTER` iterations and overshoot (overfit) the data far below vanilla's
stopping misfit, while vanilla stops early at its target. **Fair** = time for each to reach the *same*
data misfit (vanilla's final phi_d) with a comparable model — the honest apples-to-apples speedup.

In [ ]:
phi_target = res_gn['phi_d'][-1]                      # data misfit vanilla actually reached
# first pytorch outer that reaches that misfit (its time-to-equivalent-solution)
pt_phi, pt_t = res_pt['phi_d'], res_pt['times']
if np.any(pt_phi <= phi_target):
    k = int(np.argmax(pt_phi <= phi_target))
    t_pt_eq, err_pt_eq = float(pt_t[k]), errf(res_pt['models'][k+1])
else:
    k = len(pt_phi)-1; t_pt_eq, err_pt_eq = float(pt_t[-1]), errf(res_pt['model'])

print(f"Vanilla GN-Pardiso (CPU):   {res_gn['time']:6.1f} s   phi_d {res_gn['phi_d'][-1]:7.1f}   err {errf(res_gn['model']):.3f}   ({len(res_gn['phi_d'])} iters)")
print(f"PyTorch L-BFGS-cuDSS (GPU): {res_pt['time']:6.1f} s   phi_d {res_pt['phi_d'][-1]:7.1f}   err {errf(res_pt['model']):.3f}   VRAM {res_pt['vram']:.2f} GB   ({len(res_pt['phi_d'])} outer, forced)")
print("-"*92)
print(f"RAW speedup  (full runs):              {res_gn['time']/res_pt['time']:5.2f}x   <- unfair: pytorch is forced to over-iterate & overfit")
print(f"FAIR speedup (time to phi_d <= {phi_target:5.0f}):  {res_gn['time']/t_pt_eq:5.2f}x   pytorch hits it at outer {k+1} in {t_pt_eq:.1f}s, err {err_pt_eq:.3f}")
print(f"   (at that point pytorch model err {err_pt_eq:.3f} vs vanilla {errf(res_gn['model']):.3f} -> quality-matched)")

## 7. Recovered models — True / vanilla / pytorch (Y=0 slice)

In [ ]:
fig, axs = plt.subplots(3,1,figsize=(11,10))
pslice(axs[0], m_true, "True (two spheres)")
pslice(axs[1], res_gn['model'], f"Vanilla GN-Pardiso (CPU) | err {errf(res_gn['model']):.3f}")
pslice(axs[2], res_pt['model'], f"PyTorch L-BFGS-cuDSS (GPU) | err {errf(res_pt['model']):.3f}")
for a in axs: fig.colorbar(a.collections[0], ax=a, label=r"$\rho$")
plt.tight_layout(); plt.show()

## 8. Convergence — data misfit vs iteration, **vs wall-clock time**, and model error

The middle panel (phi_d vs wall time) is the meaningful one: it shows how fast each scheme drives the
misfit down in real seconds. The GPU curve reaches vanilla's level far to the left = the fair speedup.

In [ ]:
err_gn = [errf(m) for m in res_gn['models']]
err_pt = [errf(m) for m in res_pt['models']]
phi_target = res_gn['phi_d'][-1]
fig, ax = plt.subplots(1,3, figsize=(17,4.2))
# (a) phi_d vs iteration
ax[0].semilogy(np.arange(1,len(res_gn['phi_d'])+1), res_gn['phi_d'], "o-", label="vanilla GN-Pardiso")
ax[0].semilogy(np.arange(1,len(res_pt['phi_d'])+1), res_pt['phi_d'], "s-", label="pytorch L-BFGS-cuDSS")
ax[0].axhline(target, ls="--", c="k", lw=1, label="target (chi=1)")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel(r"$\phi_d$"); ax[0].set_title("Data misfit vs iteration"); ax[0].legend()
# (b) phi_d vs wall-clock time  <- the fair comparison
ax[1].semilogy(res_gn['times'][1:], res_gn['phi_d'], "o-", label="vanilla GN-Pardiso (CPU)")
ax[1].semilogy(res_pt['times'],      res_pt['phi_d'], "s-", label="pytorch L-BFGS-cuDSS (GPU)")
ax[1].axhline(phi_target, ls=":", c="C0", lw=1.2, label=f"vanilla final phi_d={phi_target:.0f}")
ax[1].set_xlabel("wall-clock time (s)"); ax[1].set_ylabel(r"$\phi_d$")
ax[1].set_title("Data misfit vs TIME (the fair view)"); ax[1].legend(fontsize=9)
# (c) model error to true
ax[2].plot(np.arange(len(err_gn)), err_gn, "o-", label="vanilla")
ax[2].plot(np.arange(len(err_pt)), err_pt, "s-", label="pytorch")
ax[2].set_xlabel("iteration (0 = reference)"); ax[2].set_ylabel("rel. model error to true")
ax[2].set_title("Model error to true"); ax[2].legend()
plt.tight_layout(); plt.show()
print(f"final model error to true:  vanilla {err_gn[-1]:.3f}  |  pytorch {err_pt[-1]:.3f}")
print(f"NOTE pytorch min model err over iters: {min(err_pt):.3f} at iter {int(np.argmin(err_pt))}",
      "(forcing past it overfits the 10% noise)")

## 9. Interactive per-iteration viewer (Y=0 slice + data fit)

Slide `iter`: reference slice, vanilla model slice, pytorch model slice, and dobs vs predicted.
The two schemes have different iteration counts -> each panel clamps to its last available.

In [ ]:
gnm, gnd = res_gn['models'], res_gn['dpreds']
ptm, ptd = res_pt['models'], res_pt['dpreds']
n_it = max(len(gnm), len(ptm))

def view_iter(it=n_it-1):
    ig, ip = min(it, len(gnm)-1), min(it, len(ptm)-1)
    fig, axs = plt.subplots(4, 1, figsize=(11, 12), gridspec_kw={"height_ratios":[1,1,1,0.8]})
    pslice(axs[0], m_true,  "True")
    pslice(axs[1], gnm[ig], f"Vanilla GN-Pardiso  -  iter {ig}/{len(gnm)-1}  |  err {errf(gnm[ig]):.3f}")
    pslice(axs[2], ptm[ip], f"PyTorch L-BFGS-cuDSS  -  iter {ip}/{len(ptm)-1}  |  err {errf(ptm[ip]):.3f}")
    for a in axs[:3]: fig.colorbar(a.collections[0], ax=a, label=r"$\rho$")
    axs[3].plot(dobs, "k.", ms=3, label="dobs")
    axs[3].plot(gnd[ig], "C0-", lw=1.0, label=f"dpred vanilla (it {ig})")
    axs[3].plot(ptd[ip], "C1-", lw=1.0, label=f"dpred pytorch (it {ip})")
    axs[3].set_xlabel("data index"); axs[3].set_ylabel("V"); axs[3].set_title("Observed vs predicted")
    axs[3].legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.show()

if HAS_WIDGETS:
    widgets.interact(view_iter, it=widgets.IntSlider(min=0, max=n_it-1, step=1, value=n_it-1, description="iter"))
else:
    print("ipywidgets not available -> static last iteration."); view_iter(n_it-1)